# Logit lens on GPT-2 XL — dataset evaluation

The handwritten logit lens (see `notebooks/logit_lens/logit_lens_examples.ipynb`) over all six sets of `data/evaluations`,
and everything the data says about it: whether GPT-2 XL solves the tasks, whether the intermediate concepts
show up in the lens above chance, at what depth, what drives a hit, and how the lens behaves in general.

**Protocol** (`data/evaluations/README.md`):

* **readout position** — one token per prompt: the last prompt token (the one right before `target`,
  the closing period, the tail of the typo); for poetry — the newline at the end of the first line;
* **rank** — 1-based rank of a word among all 50 257 tokens at that position, min over its single-token
  spellings (with/without leading space, case); for order-ops — min over the synonym set
  (numbers → digits and words, operations → symbols and words); multi-token words fall back to their first token;
* **pass@k** — mean over items of the fraction of `intermediates` whose min-over-layers rank ≤ k;
* **control** — the same ranks for the intermediates of *another* item of the same dataset.
  It is the chance level: a lens that surfaces any vaguely topical word scores here too.

## Setup

In [ ]:
import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/ai360-project-school-j-lens/jacobian-lens-gpt2.git"
REPO_DIR = os.path.abspath("jacobian-lens-gpt2")

# ноутбук открыт из клона репозитория (notebooks/<lens>/) — ищем его корень выше по дереву
local_repo = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "gpt2" / "lens_eval.py").is_file()), None)
if local_repo is not None:
    REPO_DIR = str(local_repo)
elif os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
sys.path.append(REPO_DIR)

In [ ]:
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from gpt2 import GPT2LensModel
from gpt2.lens_eval import (
    DATASETS,
    ROLE_NAMES,
    evaluate_all,
    first_layer,
    layer_hit_rate,
    layer_mean,
    layer_median_rank,
    load_eval,
    logit_lens,
    pass_at_k,
    plot_layer_curves,
    shown_layers,
)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

In [ ]:
MODEL_ID = "openai-community/gpt2-xl"  # самый большой GPT-2: 1.5B параметров, 48 слоёв, d_model 1600
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

gpt = GPT2LensModel.from_pretrained(MODEL_ID, device=DEVICE)
logit = partial(logit_lens, gpt)
LAYER_STRIDE = 4  # в визуализациях и таблицах по слоям: каждый 4-й слой + выход модели
gpt

In [ ]:
evals = {dataset: load_eval(REPO_DIR, dataset) for dataset in DATASETS}

K = 10  # главный порог ранга для hit / pass@k
LAST = gpt.n_layers - 1  # последний слой — выход модели
{dataset: len(items) for dataset, items in evals.items()}

## 1. Run

`words` — a row per (item, word) with the rank at every layer; `items` — a row per item with the model answer,
the lens top-1 at the readout position and per-layer statistics over all prompt positions.

In [ ]:
words, items = evaluate_all(gpt, logit, evals)

In [ ]:
words.head()

In [ ]:
items.head()

In [ ]:
inter = words[words.kind == "intermediate"]
control = words[words.kind == "control"]
target = words[words.kind == "target"]

## 2. Can GPT-2 XL solve the tasks at all?

The lens can only show an intermediate step the model actually computes. For the three sets with a `target`,
check whether the model's own top-1 at the readout position is the target.

In [ ]:
with_target = items.dropna(subset=["model_correct"]).astype({"model_correct": bool})
accuracy = with_target.groupby("dataset").model_correct.mean()
accuracy.to_frame("model top-1 == target")

In [ ]:
with_target.groupby("dataset").head(6)[["dataset", "item", "prompt", "target", "model_top1", "model_correct"]]

## 3. pass@k: intermediates vs control vs target

In [ ]:
scores = pass_at_k(words)
scores.pivot_table(index="dataset", columns=["kind", "k"], values="score").round(3)

The min over layers includes the last row — the model's own output. Split it: what do the inner layers (all but the last)
show that the output does not?

In [ ]:
inner = pass_at_k(words, layers=slice(None, -1)).assign(readout=f"layers 0..{LAST - 1}")
output = pass_at_k(words, layers=[LAST]).assign(readout="model output")
split = pd.concat([inner, output])
split[split.k == K].pivot_table(index="dataset", columns=["kind", "readout"], values="score").round(3)

In [ ]:
at_k = scores[scores.k == K].pivot(index="dataset", columns="kind", values="score")
at_k[["intermediate", "control", "target"]].plot.bar(figsize=(10, 4), rot=0, title=f"Logit lens: pass@{K} (min over all layers)")
plt.ylabel(f"pass@{K}")
plt.show()

## 4. At what depth do the words surface?

In [ ]:
plot_layer_curves(
    {"intermediate": layer_hit_rate(inter, K), "control": layer_hit_rate(control, K), "target": layer_hit_rate(target, K)},
    title=f"Logit lens: share of words with rank ≤ {K} at the readout position",
    ylabel=f"rank ≤ {K}",
)
plot_layer_curves(
    {"intermediate": layer_median_rank(inter), "control": layer_median_rank(control), "target": layer_median_rank(target)},
    title="Logit lens: median rank at the readout position",
    ylabel="median rank",
    logy=True,
)

Where the best rank of each intermediate is reached (only words that are hits, rank ≤ K):

In [ ]:
(
    inter[inter.best_rank <= K]
    .groupby("dataset").best_layer.value_counts().unstack(fill_value=0)
    .reindex(columns=range(gpt.n_layers), fill_value=0)
)

## 5. How the logit lens behaves in general

Over all prompt positions (not only the readout one):

* **agreement** — the layer's top-1 equals the model's top-1: how deep the final prediction is formed;
* **copy rate** — the layer's top-1 is the input token at that position: the residual stream still "is" the token embedding
  (GPT-2 ties `wte` and `lm_head`, so an untouched embedding decodes to itself);
* **KL(model ‖ layer)** — how far the layer's distribution is from the output one.

In [ ]:
agreement = layer_mean(items, "agreement")
copy_rate = layer_mean(items, "copy_rate")
kl = layer_mean(items, "kl_to_final")

plot_layer_curves(
    {"top-1 == model top-1": agreement, "top-1 == input token": copy_rate},
    title="Logit lens over all prompt positions",
    ylabel="share of positions",
)
plot_layer_curves({"KL(model || layer)": kl}, title="Logit lens: KL to the model output", ylabel="nats")

In [ ]:
pd.DataFrame({
    "agreement ≥ 50% from": {d: first_layer(agreement.loc[d], 0.5) for d in agreement.index},
    "KL ≤ 1 nat from": {d: first_layer(kl.loc[d], 1.0, above=False) for d in kl.index},
    "copy rate at L0": copy_rate[0],
    f"copy rate at L{LAST - 1}": copy_rate[LAST - 1],
}).round(3)

## 6. What drives a hit

### 6a. The word is already in the prompt

Early layers of the logit lens decode the input token itself, so a word that is literally in the prompt
(e.g. a number in order-ops) can be a "hit" without any computation.

In [ ]:
hits = inter.assign(hit=inter.best_rank <= K)
hits.pivot_table(index="dataset", columns="in_prompt", values="hit", aggfunc=["mean", "size"]).round(3)

### 6b. Single-token words vs words that only have a first-token fallback

In [ ]:
hits.pivot_table(index="dataset", columns="single_token", values="hit", aggfunc=["mean", "size"]).round(3)

### 6c. Role of the intermediate

multilingual: `[language, relation, source word, answer in English]`; order-ops: `[intermediate number, operation]`.

In [ ]:
roles = hits[hits.dataset.isin(list(ROLE_NAMES))].copy()
roles["role_name"] = [ROLE_NAMES[d][r] for d, r in zip(roles.dataset, roles.role)]
roles.groupby(["dataset", "role_name"]).agg(
    hit_rate=("hit", "mean"), median_best_rank=("best_rank", "median"), n=("hit", "size")
).round(3)

In [ ]:
fig, axes = plt.subplots(1, len(ROLE_NAMES), figsize=(12, 3.5), sharey=True)
for ax, (dataset, group) in zip(axes, roles.groupby("dataset")):
    for name, role_group in group.groupby("role_name"):
        ax.plot((np.stack(role_group.ranks) <= K).mean(0), marker="o", ms=3, label=name)
    ax.set_title(dataset)
    ax.set_xlabel("layer (last = model output)")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
axes[0].set_ylabel(f"rank ≤ {K}")
plt.show()

### 6d. Does the intermediate show up more often when the model gets the answer right?

In [ ]:
correctness = hits.merge(with_target[["dataset", "item", "model_correct"]], on=["dataset", "item"])
correctness.pivot_table(index="dataset", columns="model_correct", values="hit", aggfunc=["mean", "size"]).round(3)

## 7. Items: successes and failures

In [ ]:
columns = ["dataset", "item", "word", "best_rank", "best_layer", "in_prompt"]
inter.sort_values("best_rank").groupby("dataset").head(3)[columns]

In [ ]:
inter.sort_values("best_rank", ascending=False).groupby("dataset").head(3)[columns]

What the lens reads at the readout position, layer by layer (first items of each dataset):

In [ ]:
readout = items.groupby("dataset").head(2).set_index(["dataset", "item"]).readout_top1.apply(pd.Series)
readout.columns.name = "layer"
readout[shown_layers(gpt.n_layers, LAYER_STRIDE)]

The most common readout top-1 tokens per layer over all items — generic tokens mean the lens shows no item-specific content there:

In [ ]:
top1_by_layer = pd.DataFrame(items.readout_top1.tolist())
pd.DataFrame({
    "distinct top-1 / items": top1_by_layer.nunique() / len(top1_by_layer),
    "most common": {
        layer: ", ".join(f"{token!r}×{count}" for token, count in column.value_counts().head(5).items())
        for layer, column in top1_by_layer.items()
    },
}).rename_axis("layer")

## 8. Conclusions

The cell below turns the numbers above into statements, so they stay correct after a re-run
(other seed, other `K`, other model size).

In [ ]:
def pct(x):
    return "n/a" if x is None or pd.isna(x) else f"{100 * x:.0f}%"


def layer(x):
    return "never" if x is None else f"L{x}"


def hit_rates(frame, column):
    """Hit rate (best rank <= K) split by a boolean column: {value: (rate, n)}."""
    hit = (frame.best_rank <= K).groupby(frame[column])
    return {value: (rate, n) for value, rate, n in zip(hit.mean().index, hit.mean(), hit.size())}

In [ ]:
inner_at_k = inner[inner.k == K].pivot(index="dataset", columns="kind", values="score")
output_at_k = output[output.k == K].pivot(index="dataset", columns="kind", values="score")
inter_curve = layer_hit_rate(inter, K)
target_curve = layer_hit_rate(target, K)
lines = []

lines.append("Model accuracy (top-1 == target): " + ", ".join(f"{d} {pct(a)}" for d, a in accuracy.items()) + ".")

for dataset in at_k.index:
    i, c = at_k.loc[dataset, "intermediate"], at_k.loc[dataset].get("control")
    verdict = "clearly above" if i >= c + 0.1 else "slightly above" if i > c else "NOT above"
    peak = inter_curve.loc[dataset].iloc[:-1]
    lines.append(
        f"{dataset}: intermediates pass@{K} {pct(i)} vs control {pct(c)} — {verdict} chance; "
        f"inner layers alone {pct(inner_at_k.loc[dataset, 'intermediate'])}, model output alone "
        f"{pct(output_at_k.loc[dataset, 'intermediate'])}; best inner layer L{peak.idxmax()} "
        f"({pct(peak.max())} of words at rank ≤ {K})."
    )

for dataset in target_curve.index:
    curve = target_curve.loc[dataset]
    if curve.iloc[-1] == 0:
        lines.append(f"{dataset}: the target is never in the output top-{K}.")
    else:
        lines.append(
            f"{dataset}: the target is in the output top-{K} for {pct(curve.iloc[-1])} of items "
            f"and reaches half of that by {layer(first_layer(curve, curve.iloc[-1] / 2))}."
        )

for dataset in agreement.index:
    lines.append(
        f"{dataset}: lens top-1 == model top-1 on ≥50% of positions from {layer(first_layer(agreement.loc[dataset], 0.5))}, "
        f"KL ≤ 1 nat from {layer(first_layer(kl.loc[dataset], 1.0, above=False))}; copy rate "
        f"{pct(copy_rate.loc[dataset, 0])} at L0 → {pct(copy_rate.loc[dataset, LAST - 1])} at L{LAST - 1}."
    )

for column, yes, no in [
    ("in_prompt", "already in the prompt", "absent from it"),
    ("single_token", "single-token", "first-token fallback"),
]:
    rates = hit_rates(inter, column)
    (r_yes, n_yes), (r_no, n_no) = rates.get(True, (None, 0)), rates.get(False, (None, 0))
    lines.append(f"Intermediates {yes}: hit rate {pct(r_yes)} (n={n_yes}) vs {pct(r_no)} (n={n_no}) {no}.")

for dataset, group in roles.groupby("dataset"):
    by_role = group.groupby("role_name").hit.mean().sort_values(ascending=False)
    lines.append(f"{dataset} by role: " + ", ".join(f"{name} {pct(rate)}" for name, rate in by_role.items()) + ".")

by_correct = correctness.groupby("model_correct").hit.mean()
lines.append(
    f"Intermediate hit rate when the model is right: {pct(by_correct.get(True))}, when wrong: {pct(by_correct.get(False))}."
)

for line in lines:
    print("•", line)

### How to read these numbers

* **Intermediates vs control.** pass@k only means something relative to the control row: if intermediates are not above control,
  the lens does not reveal that latent step on GPT-2 — or the 1.5B model never computes it (compare with the accuracy in §2).
* **Inner layers vs model output.** A hit that appears only in the output row is the model *saying* the word, not thinking it.
  Evidence for a latent intermediate is a hit in the inner layers that is absent from the output.
* **Copy rate.** A high copy rate in the first layers means the logit lens there just decodes the input token; hits in those
  layers for words that are in the prompt (§6a) are copies, not reasoning.
* **Agreement / KL** show the depth at which GPT-2 XL "settles" on its prediction; before that layer the logit lens is mostly
  off-distribution for the unembedding — the gap the Jacobian lens (`notebooks/jacobian_lens/jacobian_lens.ipynb`) is designed to close.
* **Caveats.** Ranks are over 50 257 tokens; multi-token words (e.g. `español`, `português`) fall back to their first
  token, which can be a generic piece; control words come from the same dataset and therefore share its topic, so the control
  is a conservative chance level.